# Interactive Visual Lab 3: Grouped-Query Attention (GQA)

Welcome! In this interactive notebook, we explore **Grouped-Query Attention (GQA)**—the attention mechanism used in **Llama 3, Mistral, and Mirendil-Base** that reduces KV cache memory by up to **87.5%** while keeping multi-head quality.

---

## 1. Setup & Environment

In [ ]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import torch
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown
from IPython.display import display, Markdown

from src.model.attention import GroupedQueryAttention
from src.model.rope import precompute_freqs_cis

plt.rcParams['figure.dpi'] = 110
print("PyTorch and GQA module loaded successfully!")

---
## 2. Interactive Attention Head Routing: MHA vs GQA vs MQA

Select the attention mode from the dropdown below to see how Query heads map to Key/Value heads:

In [ ]:
def plot_attention_architecture(mode='GQA (12 Q : 4 KV) - Ours'):
    fig, ax = plt.subplots(figsize=(10, 5))
    
    n_q = 12
    if 'MHA' in mode:
        n_kv = 12
        title = "Multi-Head Attention (MHA): 1:1 Query to KV Mapping"
        mem_pct = "100% (Baseline)"
    elif 'GQA' in mode:
        n_kv = 4
        title = "Grouped-Query Attention (GQA): 3 Query Heads Share 1 KV Head"
        mem_pct = "33.3% (66.7% Memory Saved!)"
    else: # MQA
        n_kv = 1
        title = "Multi-Query Attention (MQA): All Query Heads Share 1 KV Head"
        mem_pct = "8.3% (91.7% Memory Saved)"
        
    # Coordinates
    y_q = np.linspace(1, 0, n_q)
    y_kv = np.linspace(1, 0, n_kv)
    
    # Draw routing lines
    ratio = n_q // n_kv
    for q_idx in range(n_q):
        kv_idx = q_idx // ratio
        ax.plot([0.3, 0.7], [y_q[q_idx], y_kv[kv_idx]], color='#58a6ff', alpha=0.45, lw=1.5)
        
    # Draw Q boxes (Blue)
    for i, y in enumerate(y_q):
        ax.plot(0.3, y, 's', color='#58a6ff', markersize=14)
        ax.text(0.24, y, f"Q{i}", ha='right', va='center', fontsize=9, fontweight='bold', color='#58a6ff')
        
    # Draw KV boxes (Purple)
    for j, y in enumerate(y_kv):
        ax.plot(0.7, y, 'o', color='#bc8cff', markersize=16)
        ax.text(0.76, y, f"KV{j}", ha='left', va='center', fontsize=9, fontweight='bold', color='#bc8cff')
        
    ax.text(0.3, 1.1, f"Query Heads ({n_q})", ha='center', fontsize=11, fontweight='bold', color='#58a6ff')
    ax.text(0.7, 1.1, f"Key/Value Heads ({n_kv})", ha='center', fontsize=11, fontweight='bold', color='#bc8cff')
    
    ax.set_xlim(0.1, 0.9)
    ax.set_ylim(-0.1, 1.2)
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=15)
    
    plt.tight_layout()
    plt.show()
    
    display(Markdown(f"""
> ### Architecture Summary for `{mode}`:
> - **Query Heads:** `{n_q}`
> - **Key/Value Heads:** `{n_kv}`
> - **KV Cache Memory Footprint:** `{mem_pct}`
> - **Inference Throughput:** Much higher on long contexts because memory bandwidth bottleneck is removed!
"""))

interact(
    plot_attention_architecture,
    mode=Dropdown(
        options=['MHA (12 Q : 12 KV)', 'GQA (12 Q : 4 KV) - Ours', 'MQA (12 Q : 1 KV)'],
        value='GQA (12 Q : 4 KV) - Ours',
        description='Attention Mode:'
    )
);

---
## 3. Real PyTorch GQA Forward Pass

Let's test our `GroupedQueryAttention` layer with `n_heads=12` and `n_kv_heads=4`:

In [ ]:
dim = 768
n_heads = 12
n_kv_heads = 4
seq_len = 16
batch_size = 2

gqa = GroupedQueryAttention(dim=dim, n_heads=n_heads, n_kv_heads=n_kv_heads)
cos, sin = precompute_freqs_cis(dim // n_heads, max_seq_len=seq_len)

x = torch.randn(batch_size, seq_len, dim)
out = gqa(x, cos, sin)

print(f"Input shape:        {list(x.shape)}")
print(f"Attention output:   {list(out.shape)}")
assert out.shape == x.shape
print("\nSUCCESS: GQA Forward pass with SDPA/FlashAttention executed cleanly!")